# Self-play analysis

Finished self-play matches, read from the server by `dc4learning.data.games` (nothing is stored locally).
Times from the server are UTC.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from dc4learning import data, stats
from dc4learning.encoding import N_SCORE, POLICY_ANGLE, POLICY_SPEED, SCORE_OFFSET, SPEEDS

index = data.games()  # e.g. data.games(since='2026-11-10T10:03') for recent games only
print(len(index), 'matches')
index.tail()

## Throughput

In [ ]:
per_hour = index.set_index('finished_at').resample('1h').size()
fig, ax = plt.subplots(figsize=(10, 2.5))
per_hour.plot(ax=ax, drawstyle='steps-post')
ax.set_ylabel('matches / hour')
duration = (index['finished_at'] - index['started_at']).dt.total_seconds() / 60
print(f'median {duration.median():.1f} min per match')

## Results

Per pair of client images (from the match names). For two different images, `p (A = B)` is a two-sided
binomial test of equal strength, with each image playing both sides.

In [ ]:
print(stats.report(index))

## Points per end (hammer team's net score)

In [ ]:
rows = []
for _, m in index.iterrows():
    meta, states = data.load_match(m['match_id'])
    s = data.match_samples(meta, states)
    for end, score in sorted(set(zip(s['end'], s['score']))):
        rows.append({'match_id': m['match_id'], 'end': end, 'hammer_net': score - SCORE_OFFSET})
ends = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(6, 3))
counts = ends['hammer_net'].value_counts().reindex(range(-SCORE_OFFSET, SCORE_OFFSET + 1), fill_value=0)
ax.bar(counts.index, counts / counts.sum())
ax.set_xlabel("hammer team's net points"); ax.set_ylabel('fraction of ends')
print('blank ends', f"{(ends['hammer_net'] == 0).mean():.1%}", '| hammer scores 2+', f"{(ends['hammer_net'] >= 2).mean():.1%}",
      '| steals', f"{(ends['hammer_net'] < 0).mean():.1%}")

## Win rate by score difference and ends left

Like `win_table.csv` in the client: rows = hammer team's lead before the end, columns = ends left.
Few games so far, so most cells are empty or noisy.

In [ ]:
rows = []
for _, m in index.iterrows():
    meta, _ = data.load_match(m['match_id'])
    s0, s1 = meta['score0'], meta['score1']
    _, states = data.load_match(m['match_id'])
    match = meta['match']
    by_id = {match['first_team_id']: 'team0', match['second_team_id']: 'team1'}
    for end in range(min(len(s0), meta['standard_end_count'])):
        first = states[(states['end_number'] == end) & (states['total_shot_number'] == 0)]
        if first.empty:
            continue
        hammer = 'team1' if by_id[first.iloc[-1]['next_shot_team_id']] == 'team0' else 'team0'
        lead0 = sum(s0[:end]) - sum(s1[:end])
        lead = lead0 if hammer == 'team0' else -lead0
        rows.append({'lead': int(np.clip(lead, -9, 9)), 'ends_left': meta['standard_end_count'] - end,
                     'won': meta['winner'] == hammer})
table = pd.DataFrame(rows)
win = table.pivot_table(index='lead', columns='ends_left', values='won', aggfunc='mean').sort_index(ascending=False)
n = table.pivot_table(index='lead', columns='ends_left', values='won', aggfunc='size').reindex_like(win)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(win, cmap='coolwarm', vmin=0, vmax=1)
ax.set_xticks(range(win.shape[1]), win.columns); ax.set_yticks(range(win.shape[0]), win.index)
ax.set_xlabel('ends left (including this one)'); ax.set_ylabel("hammer team's lead")
for i in range(win.shape[0]):
    for j in range(win.shape[1]):
        if not np.isnan(n.values[i, j]):
            ax.text(j, i, int(n.values[i, j]), ha='center', va='center', fontsize=6)
fig.colorbar(im, ax=ax, label="hammer team's win rate")

## Shots played

Policy grid: rotation x speed x angle, as the client's policy head. Speeds above 2.6 m/s are takeouts.

In [ ]:
ds_path = data.build_dataset('analysis')
ds = np.load(ds_path)
played = np.bincount(ds['policy'][ds['policy'] >= 0], minlength=2 * POLICY_SPEED * POLICY_ANGLE).reshape(2, POLICY_SPEED, POLICY_ANGLE)
fig, axs = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for r, ax in enumerate(axs):
    ax.imshow(played[r], aspect='auto', cmap='viridis')
    ax.set_title(f'rotation {r}'); ax.set_xlabel('angle index')
axs[0].set_yticks(range(POLICY_SPEED), [f'{v:.3g}' for v in SPEEDS]); axs[0].set_ylabel('speed (m/s)')
print(f"{(played > 0).sum()} distinct shots out of {played.size}")

## A model on these games

Policy: how often the played shot is in the model's top 1 / top 5 (legal shots only, as in the client).
Score: cross-entropy and accuracy of the end-score head. Point `MODEL` at a `traced.pt` or a training checkpoint.

In [ ]:
from dc4learning.model import load
from dc4learning.train import evaluate, load_dataset

MODEL = '../data/traced-a604b03.pt'  # e.g. '../runs/<name>/checkpoint-024.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load(MODEL).to(device)
d = load_dataset(ds_path, device)
metrics = evaluate(model, d, torch.arange(len(d['shot']), device=device), 1024, 1.0, device.type == 'cuda')
pd.Series(metrics).round(3)